# Manufacturing OEE Root-Cause: *The $500K Press That Wasn't the Problem*

**Vibe Analytics demo — plastics plant, extrusion blow molding of HDPE engine-oil bottles.**

> The plant manager is about to sign off on a **new \$500,000 blow molder** to fix chronic
> low OEE on press **BM-07**. Before we spend the money, let's *reason through the data* with
> an LLM as our analyst — and let's not assume the obvious answer is the right one.

This notebook is the technical backbone. The **talk track and the CRIT prompts** live in
[`README.md`](./README.md). Load the data first with [`data-loading.md`](./data-loading.md).

**OEE = Availability × Performance × Quality**
- *Availability* = run time ÷ planned time (downtime hurts this)
- *Performance* = actual output ÷ (run time × ideal rate) (slow cycles hurt this)
- *Quality* = good units ÷ total units (scrap hurts this)

The arc: **naive answer (bad machine) → LLM hypotheses → historian symptoms → the process
engineer's spreadsheet → the real root cause (regrind).** One cause drives *all three* OEE losses.

## 0. Load the tables

Six Delta tables should already exist in your lakehouse (see `data-loading.md`).

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
from pyspark.sql import functions as F

orders    = spark.table("production_orders")
downtime  = spark.table("downtime_events")
machines  = spark.table("machines")
lots      = spark.table("material_lots")
telemetry = spark.table("process_telemetry")
qc_log    = spark.table("material_qc_log")   # the process engineer's Excel

for name in ["production_orders","downtime_events","machines","material_lots",
             "process_telemetry","material_qc_log"]:
    print(f"{name:22s} {spark.table(name).count():>5} rows")

display(orders.limit(5))

## 1. The naive view: OEE by machine

We compute OEE straight from the raw counts. This is exactly the chart the plant would
put in front of leadership.

In [ ]:
oee = (orders
    .withColumn("availability", F.col("run_minutes") / F.col("planned_minutes"))
    .withColumn("performance", F.col("total_units") /
                               (F.col("run_minutes") * F.col("ideal_units_per_min")))
    .withColumn("quality",     F.col("good_units") / F.col("total_units"))
    .withColumn("oee",         F.col("availability") * F.col("performance") * F.col("quality")))

by_machine = (oee.groupBy("machine_id")
    .agg(F.round(F.avg("availability"),3).alias("availability"),
         F.round(F.avg("performance"),3).alias("performance"),
         F.round(F.avg("quality"),3).alias("quality"),
         F.round(F.avg("oee"),3).alias("oee"))
    .orderBy("oee")).toPandas()

display(by_machine)

ax = by_machine.plot.bar(x="machine_id", y="oee", legend=False, color="#c0504d")
ax.axhline(0.85, color="green", linestyle="--", label="world-class 85%")
ax.set_ylabel("OEE"); ax.set_title("OEE by machine — BM-07 looks like the problem child")
ax.legend(); plt.tight_layout(); plt.show()

**BM-07 is the worst.** The knee-jerk conclusion: *the machine is worn out — replace it.*

## 2. The downtime Pareto

Where are BM-07's minutes going? Let's Pareto the downtime reason codes.

In [ ]:
pareto = (downtime.filter(F.col("machine_id")=="BM-07")
    .groupBy("reason_code")
    .agg(F.sum("duration_min").alias("downtime_min"))
    .orderBy(F.desc("downtime_min"))).toPandas()

display(pareto)

ax = pareto.plot.bar(x="reason_code", y="downtime_min", legend=False, color="#4f81bd")
ax.set_ylabel("downtime minutes"); ax.set_title("BM-07 downtime by reason — 'MACHINE_FAULT' dominates")
plt.tight_layout(); plt.show()

`MACHINE_FAULT` dominates. Case closed, right? The data *says* it's the machine.

> **Stop.** This is the trap. Reason codes are entered by operators under pressure. Trusting
> them blindly is *automation bias*. Before we spend \$500K, let's use the LLM to force ourselves
> to think a meta-layer higher — and to actively try to **disprove** the obvious answer.

## 3. Test the first LLM hypothesis: *"is it just an old, worn-out machine?"*

See the CRIT prompt in the README. The LLM's top hypothesis is usually **asset age / condition**.
Let's test it: if age drives OEE, older machines should be worse.

In [ ]:
age = (by_machine.merge(machines.toPandas()[["machine_id","install_year","model"]],
                        on="machine_id")
       .sort_values("install_year"))
display(age[["machine_id","install_year","model","oee"]])

**The age hypothesis collapses.** BM-07 is one of the *newest* machines (2021), yet it has the
worst OEE — while BM-12 (2015, the *oldest*) is one of the best. Age is not the driver.

So we reject hypothesis #1. Good. Now bring in data the naive analysis ignored.

## 4. Symptoms from the process historian (MES / PI-AVEVA)

Let's look at *how the process behaved*, not just how much it stopped. Plot daily
bottle-weight variation and melt-pressure instability per machine.

In [ ]:
tele = (telemetry
    .withColumn("date", F.to_date("ts"))
    .groupBy("date","machine_id")
    .agg(F.round(F.avg("bottle_weight_std_g"),3).alias("weight_var"),
         F.round(F.avg("melt_pressure_std"),2).alias("melt_press_var"),
         F.round(F.avg("cycle_time_s"),2).alias("cycle_time_s"))
    .orderBy("date")).toPandas()
tele["date"] = pd.to_datetime(tele["date"])

fig, ax = plt.subplots(figsize=(11,4))
for m, g in tele.groupby("machine_id"):
    ax.plot(g["date"], g["weight_var"], marker=".", label=m)
ax.set_title("Bottle-weight variation by machine — something erupts Jun 10–18 on BM-07 / BM-09")
ax.set_ylabel("avg bottle weight std (g)"); ax.legend(); plt.tight_layout(); plt.show()

There it is: a **window of instability (roughly Jun 10–18)** on **BM-07 and BM-09** — bottle
weight swings, melt pressure gets erratic, cycles slow down. The other two machines are calm.

The historian tells us *what* is happening (unstable melt → sag, blowouts, scrap) but not *why*.
Notice BM-07 and BM-09 move **together** — that's a clue the cause is *shared*, not per-machine.

## 5. OEE over time — the window is unmistakable

In [ ]:
daily = (oee.groupBy("date","machine_id")
    .agg(F.round(F.avg("oee"),3).alias("oee"))
    .orderBy("date")).toPandas()
daily["date"] = pd.to_datetime(daily["date"])

fig, ax = plt.subplots(figsize=(11,4))
for m, g in daily.groupby("machine_id"):
    ax.plot(g["date"], g["oee"], marker=".", label=m)
ax.axvspan(pd.Timestamp("2026-06-10"), pd.Timestamp("2026-06-18"),
           color="orange", alpha=0.15, label="incident window")
ax.axhline(0.85, color="green", linestyle="--")
ax.set_title("Daily OEE — BM-07 & BM-09 crater only during the window; BM-04 & BM-12 never flinch")
ax.set_ylabel("OEE"); ax.legend(ncol=3); plt.tight_layout(); plt.show()

**BM-07 and BM-09 are perfectly healthy *except* during Jun 10–18.** A worn-out machine doesn't
heal itself. Something *external and time-bound* hit these two machines together. What do BM-07
and BM-09 share that BM-04 and BM-12 don't?

## 6. The process engineer's spreadsheet (the smoking gun)

BM-07 and BM-09 are both fed by regrind blender **BL-2**. The historian never captured regrind —
that lives only in the process engineer's Excel log. Let's bring it in.

In [ ]:
qc = qc_log.toPandas()
qc["date"] = pd.to_datetime(qc["date"])

fig, ax = plt.subplots(figsize=(11,4))
for b, g in qc.groupby("blender_id"):
    ax.plot(g["date"], g["regrind_pct_actual"], marker="o", label=f"{b} actual")
ax.axhline(qc["regrind_pct_target"].iloc[0], color="black", linestyle=":", label="target 15%")
ax.axvspan(pd.Timestamp("2026-06-10"), pd.Timestamp("2026-06-18"), color="orange", alpha=0.15)
ax.set_title("Regrind % by blender — BL-2 blew past target exactly during the window")
ax.set_ylabel("regrind %"); ax.legend(); plt.tight_layout(); plt.show()

# the engineer even wrote down what happened
display(qc[qc["notes"]!=""][["date","blender_id","regrind_pct_actual",
                             "regrind_source","blackspeck_count_per_kg",
                             "leak_fail_count","notes"]])

**There's the cause.** During Jun 10–18, blender **BL-2** ran regrind at **35–45%** vs a 15%
target — to hit a material-cost target — and for several days it pulled in a **contaminated black
color-changeover purge** (black specks, leak-test rejects). BL-2 feeds **BM-07 and BM-09**.

Over-blended, degraded regrind lowers melt strength → parison sag → **blowouts logged as
`MACHINE_FAULT`** (Availability), **slow cycles** as operators compensate (Performance), and
**black specks + leaks** (Quality). *One* root cause, *all three* OEE losses.

## 7. Re-segment by the real variable — the machine effect disappears

If the machine were the problem, BM-07 would be bad *all the time*. Let's split OEE inside vs.
outside the window, and roll it up by blender.

In [ ]:
oee2 = oee.join(machines.select("machine_id","blender_id"), "machine_id")
oee2 = oee2.withColumn("in_window",
    (F.col("date") >= F.lit("2026-06-10")) & (F.col("date") <= F.lit("2026-06-18")))

print("OEE by machine, INSIDE vs OUTSIDE the window:")
display(oee2.groupBy("machine_id").pivot("in_window")
        .agg(F.round(F.avg("oee"),3)).orderBy("machine_id"))

print("OEE by BLENDER, INSIDE vs OUTSIDE the window:")
display(oee2.groupBy("blender_id").pivot("in_window")
        .agg(F.round(F.avg("oee"),3)).orderBy("blender_id"))

**The reveal.** *Outside* the window, all four machines are essentially identical (~0.88) —
**BM-07 is not a bad machine.** The entire OEE gap is BL-2 during the regrind incident. The
problem was never the press; it was what we fed it.

> A \$500K new press would have run the same bad regrind and produced the same scrap.

## 8. Quantify it and prescribe the fix

In [ ]:
# Baseline (healthy) OEE from BL-2 machines outside the window
pdf = oee2.toPandas()
bl2 = pdf[pdf["blender_id"]=="BL-2"]
baseline = bl2[~bl2["in_window"]]["oee"].mean()
incident = bl2[ bl2["in_window"]]["oee"].mean()
lost_oee_pts = baseline - incident

# Lost good bottles during the window on BL-2 machines
win = pdf[(pdf["blender_id"]=="BL-2") & (pdf["in_window"])]
actual_good = win["good_units"].sum()
# what we'd have made at baseline OEE (same planned time & ideal rate)
would_be_good = ( (win["planned_minutes"] * win["ideal_units_per_min"]) * baseline ).sum()
lost_bottles = int(would_be_good - actual_good)

MARGIN_PER_BOTTLE = 0.09   # $ contribution margin, illustrative
lost_dollars = lost_bottles * MARGIN_PER_BOTTLE

print(f"Baseline BL-2 OEE (healthy)      : {baseline:5.1%}")
print(f"BL-2 OEE during incident         : {incident:5.1%}")
print(f"OEE points lost during window    : {lost_oee_pts*100:4.1f} pts")
print(f"Good bottles lost in the window  : {lost_bottles:,}")
print(f"Lost margin (@ ${MARGIN_PER_BOTTLE}/bottle) : ${lost_dollars:,.0f}")
print(f"Avoided capital expenditure      : $500,000  (the press we almost bought)")

## The story, in one breath

| Step | What we thought | What the data showed |
|---|---|---|
| 1. OEE by machine | BM-07 is our worst press | true, but only *on average* |
| 2. Downtime Pareto | It's `MACHINE_FAULT` | the codes are miscoded |
| 3. Age hypothesis | Old, worn-out machine | BM-07 is one of the *newest* |
| 4. Historian | — | weight/pressure erupt Jun 10–18 on BM-07 **and** BM-09 |
| 5. OEE over time | Chronically bad press | perfectly healthy *except* the window |
| 6. Excel regrind log | — | **BL-2 regrind at 35–45% + contaminated purge** |
| 7. Re-segment | Bad machine | machine effect vanishes; it's the **blender/regrind** |
| 8. Prescribe | Buy a \$500K press | fix the regrind spec — recover OEE, avoid the capex |

**The point of Vibe Analytics:** the LLM didn't hand us the answer — it helped us *reason*,
disprove the obvious, and pull together three systems (SAP MII, the historian, and a
spreadsheet) that no single dashboard connects. That's how you avoid a \$500,000 mistake.

### Recommended actions
1. **Cap regrind at ≤20%** on HDPE bottle grades; interlock the blender to the spec.
2. **Segregate color-changeover purge** — never feed black purge into clear/gray regrind.
3. **Fix the downtime reason codes** — add a `MATERIAL/REGRIND` code so this is visible next time.
4. **Add regrind % to the historian** so the process layer and the MES layer finally agree.
5. **Cancel the capital request.** The press was never the problem.